In [30]:
import zipfile
import os

zip_path = "/content/saas-subscriptions-v1.0.0-source-csv.zip"
extract_path = "/content/saas_data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP extracted successfully!")
print(os.listdir(extract_path))

ZIP extracted successfully!
['plans.csv', 'invoices.csv', 'accounts.csv', 'DATA_DICTIONARY.md', 'starter', 'subscriptions.csv', 'README.md', 'LICENSE.txt', 'payments.csv']


In [31]:
import pandas as pd

accounts = pd.read_csv("/content/saas_data/accounts.csv")
plans = pd.read_csv("/content/saas_data/plans.csv")
subscriptions = pd.read_csv("/content/saas_data/subscriptions.csv")
invoices = pd.read_csv("/content/saas_data/invoices.csv")
payments = pd.read_csv("/content/saas_data/payments.csv")

print("All 5 files loaded successfully!")

All 5 files loaded successfully!


# Basic dataset overview



In [32]:
datasets = {
    "accounts": accounts,
    "plans": plans,
    "subscriptions": subscriptions,
    "invoices": invoices,
    "payments": payments
}

for name, df in datasets.items():
    print(f"{name.upper()}")
    print("Rows    :", df.shape[0])
    print("Columns :", df.shape[1])
    print("-" * 40)

ACCOUNTS
Rows    : 5000
Columns : 5
----------------------------------------
PLANS
Rows    : 4
Columns : 4
----------------------------------------
SUBSCRIPTIONS
Rows    : 5000
Columns : 8
----------------------------------------
INVOICES
Rows    : 85448
Columns : 6
----------------------------------------
PAYMENTS
Rows    : 87287
Columns : 5
----------------------------------------


# Data types + missing values + duplicate rows

In [33]:
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"{name.upper()}")
    print(f"{'='*60}")

    print("\nData Types:")
    print(df.dtypes)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:", df.duplicated().sum())


ACCOUNTS

Data Types:
account_id             object
created_at             object
region                 object
segment                object
acquisition_channel    object
dtype: object

Missing Values:
account_id                0
created_at                0
region                 1247
segment                   0
acquisition_channel       0
dtype: int64

Duplicate Rows: 0

PLANS

Data Types:
plan_id            object
plan_name          object
monthly_price     float64
included_seats      int64
dtype: object

Missing Values:
plan_id           0
plan_name         0
monthly_price     0
included_seats    0
dtype: int64

Duplicate Rows: 0

SUBSCRIPTIONS

Data Types:
subscription_id     object
account_id          object
plan_id             object
started_at          object
ended_at            object
status              object
seats                int64
mrr                float64
dtype: object

Missing Values:
subscription_id       0
account_id            0
plan_id               0
started_at

In [34]:
pd.crosstab(
    subscriptions["status"],
    subscriptions["ended_at"].isna()
)

ended_at,False,True
status,,
active,0,4035
cancelled,965,0


###ended_at ka NULL hona is dataset mein active subscription ke saath consistently associated hai. Isliye 4035 NULL values ko fill nahi karna hai.

In [35]:
# Check missing region by customer segment

accounts[accounts["region"].isna()]["segment"].value_counts()

,count
segment,
SMB,864
mid_market,308
enterprise,75


In [36]:
accounts[accounts["region"].isna()][["account_id", "created_at", "segment", "acquisition_channel"]].head(20)

,account_id,created_at,segment,acquisition_channel
6,A000007,2023-10-24,SMB,outbound
9,A000010,2023-12-30,SMB,outbound
10,A000011,2025-07-03,SMB,outbound
15,A000016,2023-04-09,SMB,paid_search
19,A000020,2023-05-11,mid_market,referral
30,A000031,2024-11-29,SMB,referral
36,A000037,2025-06-06,SMB,paid_search
39,A000040,2024-02-15,SMB,partner
41,A000042,2023-01-24,SMB,organic
51,A000052,2024-01-04,enterprise,organic


In [37]:
accounts[accounts["region"].isna()].shape

(1247, 5)

In [38]:
accounts["region"].value_counts(dropna=False)

,count
region,
EMEA,1259
LATAM,1250
NaN,1247
APAC,1244


#Data Cleaning

In [39]:
accounts["created_at"] = pd.to_datetime(accounts["created_at"])

subscriptions["started_at"] = pd.to_datetime(subscriptions["started_at"])
subscriptions["ended_at"] = pd.to_datetime(subscriptions["ended_at"])

invoices["invoice_date"] = pd.to_datetime(invoices["invoice_date"])
invoices["due_date"] = pd.to_datetime(invoices["due_date"])

payments["attempted_at"] = pd.to_datetime(payments["attempted_at"])

print("Date conversion completed!")

Date conversion completed!


In [40]:
print("Before:", accounts["region"].isna().sum())

accounts["region"] = accounts["region"].fillna("Unknown")

print("After:", accounts["region"].isna().sum())

Before: 1247
After: 0


#Validation after cleaning

In [41]:
print("Accounts rows:", len(accounts))
print("Unique account IDs:", accounts["account_id"].nunique())
print("Missing region:", accounts["region"].isna().sum())
print("\nRegions:")
print(accounts["region"].value_counts())

region
EMEA       1259
LATAM      1250
Unknown    1247
APAC       1244
Name: count, dtype: int64


In [42]:
pd.crosstab(
    subscriptions["status"],
    subscriptions["ended_at"].isna()
)

ended_at,False,True
status,,
active,0,4035
cancelled,965,0


In [43]:
print("Duplicate subscription_id:",
      subscriptions["subscription_id"].duplicated().sum())

print("Missing account_id:",
      subscriptions["account_id"].isna().sum())

print("Missing plan_id:",
      subscriptions["plan_id"].isna().sum())

print("Invalid seats:",
      (subscriptions["seats"] <= 0).sum())

print("Invalid MRR:",
      (subscriptions["mrr"] <= 0).sum())

Duplicate subscription_id: 0
Missing account_id: 0
Missing plan_id: 0
Invalid seats: 0
Invalid MRR: 0


In [44]:
print("Invalid account_id:",
      (~subscriptions["account_id"].isin(accounts["account_id"])).sum())

print("Invalid plan_id:",
      (~subscriptions["plan_id"].isin(plans["plan_id"])).sum())

Invalid account_id: 0
Invalid plan_id: 0


###Invoice inspection

In [45]:
print("Duplicate invoice_id:",
      invoices["invoice_id"].duplicated().sum())

print("Missing subscription_id:",
      invoices["subscription_id"].isna().sum())

print("Invalid amount_due:",
      (invoices["amount_due"] <= 0).sum())

print("\nInvoice Status:")
print(invoices["invoice_status"].value_counts(dropna=False))

Duplicate invoice_id: 0
Missing subscription_id: 0
Invalid amount_due: 0

Invoice Status:
invoice_status
paid    80101
open     5311
void       36
Name: count, dtype: int64


In [46]:
print(
    "Invalid subscription_id:",
    (~invoices["subscription_id"].isin(
        subscriptions["subscription_id"]
    )).sum()
)

Invalid subscription_id: 0


In [47]:
print("Invalid due dates:",
      (invoices["due_date"] < invoices["invoice_date"]).sum())

print("Missing invoice_date:",
      invoices["invoice_date"].isna().sum())

print("Missing due_date:",
      invoices["due_date"].isna().sum())

Invalid due dates: 0
Missing invoice_date: 0
Missing due_date: 0


###PAYMENT VALIDATION

In [48]:
print("Duplicate payment_id:",
      payments["payment_id"].duplicated().sum())

print("Missing invoice_id:",
      payments["invoice_id"].isna().sum())

print("Invalid amount:",
      (payments["amount"] <= 0).sum())

print("\nPayment Status:")
print(payments["payment_status"].value_counts(dropna=False))

payment_status
succeeded    80101
failed        7186
Name: count, dtype: int64


In [49]:
print("Duplicate payment_id:",
      payments["payment_id"].duplicated().sum())

print("Missing invoice_id:",
      payments["invoice_id"].isna().sum())

print("Invalid amount:",
      (payments["amount"] <= 0).sum())

Duplicate payment_id: 0
Missing invoice_id: 0
Invalid amount: 0


In [50]:
print(
    "Invalid invoice_id:",
    (~payments["invoice_id"].isin(
        invoices["invoice_id"]
    )).sum()
)


Invalid invoice_id: 0


In [51]:
import os

processed_path = "/content/saas_data/processed"
os.makedirs(processed_path, exist_ok=True)

accounts.to_csv(f"{processed_path}/accounts_clean.csv", index=False)
plans.to_csv(f"{processed_path}/plans_clean.csv", index=False)
subscriptions.to_csv(f"{processed_path}/subscriptions_clean.csv", index=False)
invoices.to_csv(f"{processed_path}/invoices_clean.csv", index=False)
payments.to_csv(f"{processed_path}/payments_clean.csv", index=False)

print("All cleaned datasets saved successfully!")

All cleaned datasets saved successfully!


In [52]:
import shutil

shutil.make_archive(
    "/content/saas_cleaned_files",
    "zip",
    "/content/saas_data/processed"
)

print("Cleaned CSV ZIP ready!")

Cleaned CSV ZIP ready!
